<a href="https://colab.research.google.com/github/Berkaycetinn/log-analyzer/blob/main/05_llm_analiz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import json, re

with open("events.json", encoding="utf-8") as f:
    raw = json.load(f)

def strip(o):
    if isinstance(o, dict):
        return {k: strip(v) for k, v in o.items() if k != "embedding_cluster"}
    if isinstance(o, list):
        return [strip(v) for v in o]
    return o

data = strip(raw)
payload = json.dumps(data, ensure_ascii=False, indent=1)
print("Olay sayısı:", len(data["events"]), "| Payload:", len(payload), "karakter")

Olay sayısı: 3 | Payload: 18911 karakter


In [5]:
!pip install -q -U transformers accelerate

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen3-14B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto", device_map="auto")
print("Yüklendi:", MODEL, "| Cihaz:", model.device, "| Bellek: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

Loading weights:   0%|          | 0/443 [00:00<?, ?it/s]

Yüklendi: Qwen/Qwen3-14B | Cihaz: cuda:0 | Bellek: 29.5 GB


In [6]:
SYSTEM = """You are an SRE assistant that analyzes pre-processed application logs.
You receive a JSON package: log events (time-windowed groups of flagged log lines), each with templates,
counts, time ranges, raw example lines, ranges of masked values, and surrounding context.
The log comes from a single process (an application master), so lines from other components are NOT visible.

Rules:
1. Use ONLY the data in the package. Anything that depends on outside knowledge (what a port number usually means,
   how a framework typically behaves) must be listed under "assumptions", never stated as fact.
2. "evidence" quotes must be EXACT substrings copied from the package, at most 200 characters each. Never paraphrase inside a quote.
3. Confidence: "high" only if the log itself states the cause. "medium" if several consistent pieces of evidence
   point to one explanation but other explanations remain possible. "low" otherwise.
4. Severity describes the impact on the running job (none, low, medium, high), NOT the log level of the lines.
5. The sample is cut at "sample_end". Never state or imply that a problem ended, was resolved, or lasted a certain
   total duration beyond what the data shows.
6. If the data cannot answer something, say so under "unknowns". Do not fill gaps with guesses.
7. Write all explanations in Turkish. Keep quotes verbatim in their original language. Output ONLY valid JSON, no markdown.

Schema:
{
  "events": [
    {
      "event_id": <int>,
      "summary": "<1-2 sentences>",
      "severity": "none|low|medium|high",
      "likely_root_cause": "<hypothesis, or null if the data does not support one>",
      "confidence": "low|medium|high",
      "confidence_reason": "<why this level>",
      "evidence": [{"time": "<HH:MM:SS.mmm>", "quote": "<exact substring>"}],
      "assumptions": ["<anything not directly in the data>"],
      "alternative_explanations": ["..."],
      "unknowns": ["..."],
      "recommended_checks": ["<what a human should look at next>"],
      "recommended_fix": "<string or null>"
    }
  ],
  "cross_event_notes": "<relations between events, only if the data supports them>"
}"""

USER = "Analyze every event in the following package.\n\n" + payload

In [7]:
LAST_RAW = ""

def analyze():
    global LAST_RAW
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": USER}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=6000, do_sample=True,
                             temperature=0.7, top_p=0.8, top_k=20)
    gen = out[0][inputs["input_ids"].shape[1]:]
    LAST_RAW = tok.decode(gen, skip_special_tokens=True).strip()
    cleaned = re.sub(r"<think>.*?</think>", "", LAST_RAW, flags=re.S).strip()
    cleaned = cleaned[cleaned.find("{"): cleaned.rfind("}") + 1]
    usage = {"input": inputs["input_ids"].shape[1], "output": len(gen)}
    return json.loads(cleaned), usage

result, usage = analyze()
print("Token:", usage["input"], "girdi /", usage["output"], "çıktı")
print("Olay sayısı:", len(result["events"]))

Token: 8698 girdi / 1965 çıktı
Olay sayısı: 3


In [8]:
def all_strings(o):
    if isinstance(o, str):
        yield o
    elif isinstance(o, dict):
        for v in o.values():
            yield from all_strings(v)
    elif isinstance(o, list):
        for v in o:
            yield from all_strings(v)

norm = lambda s: re.sub(r"\s+", " ", s).strip().lower()
haystack = norm(" ".join(all_strings(data)))

def check_evidence(res):
    total, bad = 0, []
    for ev in res["events"]:
        for e in ev.get("evidence", []):
            total += 1
            if norm(e["quote"]) not in haystack:
                bad.append((ev["event_id"], e["quote"][:120]))
    return total, bad

total, bad = check_evidence(result)
print(f"Kanıt alıntısı: {total} | Pakette bulunamayan: {len(bad)}")
for b in bad:
    print("  olay", b[0], "->", b[1])

Kanıt alıntısı: 9 | Pakette bulunamayan: 1
  olay 2 -> Diagnostics report from attempt_1445144423722_0020_m_000002_0: Error: java.net.NoRouteToHostException: No Route to Host 


In [9]:
for ev in result["events"]:
    print("=" * 90)
    print(f"OLAY {ev['event_id']} | önem: {ev['severity']} | güven: {ev['confidence']}")
    print("Özet:", ev["summary"])
    print("Olası kök neden:", ev["likely_root_cause"])
    print("Güven gerekçesi:", ev["confidence_reason"])
    print("\nKanıtlar:")
    for e in ev["evidence"]:
        print("  ", e["time"], "|", e["quote"][:110])
    for key, title in [("assumptions", "Varsayımlar"), ("alternative_explanations", "Alternatifler"),
                       ("unknowns", "Bilinmeyenler"), ("recommended_checks", "Önerilen kontroller")]:
        print(f"\n{title}:")
        for x in ev[key]:
            print("  -", x)
    print("\nÖnerilen çözüm:", ev["recommended_fix"])

print("\n" + "=" * 90)
print("Olaylar arası notlar:", result["cross_event_notes"])

OLAY 0 | önem: medium | güven: medium
Özet: Bilinmeyen bir konteyner kimliği için 'Container complete event' hatası oluştu. Bu, konteynerin tamamlanmasıyla ilgili bir hata olabilir.
Olası kök neden: Konteynerin tamamlanması sırasında bir hata oluştu ve bu konteynerin kimliği sistemde bilinmiyor olabilir.
Güven gerekçesi: Log, konteynerin tamamlanmasıyla ilgili bir hata olduğunu ve konteynerin kimliğinin bilinmediğini belirtiyor. Ancak hata nedeni tam olarak belirsiz.

Kanıtlar:
   18:04:11.034 | Container complete event for unknown container id container_1445144423722_0020_01_000012

Varsayımlar:
  - Konteynerin kimliğinin bilinmemesi, sistemde bir kayıtlama hatası ya da konteynerin yanlışlıkla silinmesiyle ilgili olabilir.

Alternatifler:
  - Konteynerin kimliği sistemde kaydedilmediği için bu hata oluşabilir.

Bilinmeyenler:
  - Konteynerin tamamlanmasının neden olduğu hata tam olarak belirsizdir.

Önerilen kontroller:
  - Konteynerin kimliğiyle ilgili sistemdeki kayıtları kontrol ed

In [10]:
runs, failures = [], 0
for i in range(3):
    try:
        r, _u = analyze()
        runs.append(r)
        with open(f"analysis_run{i}.json", "w", encoding="utf-8") as f:
            json.dump(r, f, ensure_ascii=False, indent=2)
    except Exception as e:
        failures += 1
        print(f"Çalıştırma {i} başarısız:", type(e).__name__, "|", LAST_RAW[-200:].replace("\n", " "))

print(f"\nBaşarılı: {len(runs)}/3 | Bozuk JSON: {failures}")
if runs:
    print("\nOlay | " + " | ".join(f"çalıştırma {i}" for i in range(len(runs))))
    for k in range(len(runs[0]["events"])):
        row = [f"{r['events'][k]['severity']}/{r['events'][k]['confidence']}" for r in runs if k < len(r["events"])]
        print(f"{runs[0]['events'][k]['event_id']:>4} | " + " | ".join(row))
    print("\nBulunamayan alıntılar:", [len(check_evidence(r)[1]) for r in runs])
    print("\nOlay 2 kök neden cümleleri:")
    for i, r in enumerate(runs):
        print(f"[{i}]", r["events"][-1]["likely_root_cause"])


Başarılı: 3/3 | Bozuk JSON: 0

Olay | çalıştırma 0 | çalıştırma 1 | çalıştırma 2
   0 | medium/medium | medium/medium | medium/medium
   1 | medium/medium | low/high | medium/medium
   2 | high/high | high/medium | high/medium

Bulunamayan alıntılar: [1, 0, 0]

Olay 2 kök neden cümleleri:
[0] Düğüm ve hizmetler arası iletişim sorunları, özellikle 'No Route to Host' ve 'Failed to renew lease' hataları nedeniyle işlerin tamamlanamamasına neden olabilir.
[1] MSRA-SA-41 sunucusuna yapılan bağlantılar başarısız oldu. Bu, network veya sunucu düzeyinde bir problemi işaret ediyor olabilir.
[2] Sunucuya bağlanamama ve adres değişikliği sorunları, ağ veya sunucu konfigürasyonu ile ilgili olabilir.


In [11]:
import difflib

# 1) Pakette ne vardı?
print("Üst düzey anahtarlar:", list(data.keys()))
for e in data["events"]:
    print("olay", e["event_id"], "| anahtarlar:", list(e.keys()),
          "| related_lines:", len(e.get("related_lines", [])))

# 2) Bulunamayan alıntılar: model ne yazdı, pakette en yakın ne var?
strings = [s for s in all_strings(data) if len(s) > 20]
for i, r in enumerate(runs):
    for ev in r["events"]:
        for e in ev.get("evidence", []):
            if norm(e["quote"]) not in haystack:
                near = difflib.get_close_matches(e["quote"], strings, n=1, cutoff=0.0)
                print(f"\n[çalıştırma {i}, olay {ev['event_id']}]")
                print("  model   :", e["quote"][:160])
                print("  en yakın:", near[0][:160] if near else "-")

Üst düzey anahtarlar: ['sample_start', 'sample_end', 'n_rows_total', 'note', 'events', 'activity_per_minute', 'pre_storm_facts']
olay 0 | anahtarlar: ['event_id', 'start', 'end', 'n_rows', 'templates', 'context_before', 'preceding_lines', 'related_lines'] | related_lines: 3
olay 1 | anahtarlar: ['event_id', 'start', 'end', 'n_rows', 'templates', 'context_before', 'preceding_lines', 'related_lines'] | related_lines: 7
olay 2 | anahtarlar: ['event_id', 'start', 'end', 'n_rows', 'templates', 'context_before', 'preceding_lines', 'related_lines'] | related_lines: 0

[çalıştırma 0, olay 2]
  model   : Diagnostics report from attempt_1445144423722_0020_m_000002_0: Error: java.net.NoRouteToHostException: No Route to Host from  MININT-FNANLI5/127.0.0.1 to msra-s
  en yakın: Diagnostics report from <ID>: Error: java.net.NoRouteToHostException: No Route to Host from MININT-FNANLI5/<IP> to msra-sa-<NUM>:<NUM> failed on socket timeout 


In [12]:
def longest_found_prefix(q):
    qn = norm(q)
    lo, hi = 0, len(qn)
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if qn[:mid] in haystack:
            lo = mid
        else:
            hi = mid - 1
    return qn[:lo], qn[lo:]

for i, r in enumerate(runs):
    for ev in r["events"]:
        for e in ev.get("evidence", []):
            if norm(e["quote"]) not in haystack:
                ok, rest = longest_found_prefix(e["quote"])
                idx = haystack.find(ok)
                print(f"[çalıştırma {i}, olay {ev['event_id']}] eşleşen: {len(ok)} karakter")
                print("   modelin eşleşmeyen kuyruğu:", repr(rest))
                print("   pakette devamı            :", repr(haystack[idx + len(ok): idx + len(ok) + len(rest) + 20]))

[çalıştırma 0, olay 2] eşleşen: 250 karakter
   modelin eşleşmeyen kuyruğu: 'host: no further information; for more details see: http://wiki.apache.org/hadoop/noroutetohost'
   pakette devamı            : '18:06:26.029000 18:06:28.217000 attempt_1445144423722_0020_m_000001_0 attempt_1445144423722_0020_m_000002_0 task: <'


In [13]:
import time

def analyze(thinking=False, max_new=6000):
    global LAST_RAW
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": USER}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
    inputs = tok(text, return_tensors="pt").to(model.device)
    params = dict(temperature=0.6, top_p=0.95, top_k=20) if thinking else dict(temperature=0.7, top_p=0.8, top_k=20)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new, do_sample=True, **params)
    gen = out[0][inputs["input_ids"].shape[1]:]
    LAST_RAW = tok.decode(gen, skip_special_tokens=True).strip()
    body = LAST_RAW.split("</think>")[-1] if "</think>" in LAST_RAW else LAST_RAW
    m = re.search(r'\{\s*"events"', body)
    body = body[m.start(): body.rfind("}") + 1] if m else body[body.find("{"): body.rfind("}") + 1]
    usage = {"input": inputs["input_ids"].shape[1], "output": len(gen)}
    return json.loads(body), usage

runs_v1 = [result] + runs          # v1: 4 örnek (Hücre 3 + Hücre 6)

t = time.time()
r2, u2 = analyze(thinking=True, max_new=12000)
print(f"Süre: {time.time() - t:.0f} sn | Token: {u2}")
for ev in r2["events"]:
    print(f"\nolay {ev['event_id']} | {ev['severity']}/{ev['confidence']}")
    print("  kök neden:", ev["likely_root_cause"])
    print("  bilinmeyenler:", ev["unknowns"])

Süre: 67 sn | Token: {'input': 8694, 'output': 2232}

olay 0 | low/low
  kök neden: Konteynerin ataması sırasında bir sorun yaşanmış ve tamamlanması sırasında sistemde takip edilemeyen bir durum oluşmuş olabilir.
  bilinmeyenler: ['Konteynerin tamamlanmasının neden olduğu tam olarak belirsiz.', 'Konteynerin ataması sırasında hangi hata oluştuğu bilinmiyor.']

olay 1 | low/low
  kök neden: ApplicationMaster, konteynerin işlemi tamamlanmış veya hatalı olduğunu tespit ederek onu öldürmüş olabilir.
  bilinmeyenler: ['Konteynerin öldürülmesinin kesin nedeni belirsiz.', 'Konteynerin temizlenmesi sırasında hangi hata oluştuğu bilinmiyor.']

olay 2 | high/medium
  kök neden: Ağ bağlantısı sorunları, özellikle RM ile iletişim ve HDFS ile ilgili lease yenileme hataları. Bu sorunlar, NoRouteToHostException ile ilişkilendirilir.
  bilinmeyenler: ['Ağ sorunlarının tam olarak nedeni belirsiz.', 'HDFS yapılandırmasındaki hataların tam olarak nedeni bilinmiyor.']


In [14]:
import pandas as pd

runs_v2, fails = [r2], 0
for i in (1, 2):
    try:
        r, _ = analyze(thinking=True, max_new=12000)
        runs_v2.append(r)
    except Exception as e:
        fails += 1
        print("Bozuk JSON:", type(e).__name__)
for i, r in enumerate(runs_v2):
    with open(f"analysis_v2_run{i}.json", "w", encoding="utf-8") as f:
        json.dump(r, f, ensure_ascii=False, indent=2)

RESOLVED = re.compile(r"çözüld|çözülmüş|sona erdi|sona ermiş|düzeldi|düzelmiş|giderildi|dakika boyunca|dakika süren|dakika sürdü", re.I)

def score(r):
    ev = {e.get("event_id"): e for e in r.get("events", [])}
    g = lambda i, k: ev.get(i, {}).get(k, "-")
    txt = json.dumps([{k: v for k, v in e.items() if k != "evidence"} for e in r.get("events", [])]
                     + [r.get("cross_event_notes", "")], ensure_ascii=False)
    ev2 = " ".join(x["quote"] for x in ev.get(2, {}).get("evidence", []))
    return {
        "olay0": f"{g(0,'severity')}/{g(0,'confidence')}",
        "olay1": f"{g(1,'severity')}/{g(1,'confidence')}",
        "olay2": f"{g(2,'severity')}/{g(2,'confidence')}",
        "bulunamayan_alıntı": len(check_evidence(r)[1]),
        "çözüldü/süre_iddiası": len(RESOLVED.findall(txt)),
        "8030_ve_9000": ("8030" in txt) and ("9000" in txt),
        "NoRoute_kanıtı": "noroutetohost" in ev2.lower(),
    }

rows = []
for name, rs in [("v1", runs_v1), ("v2", runs_v2)]:
    for i, r in enumerate(rs):
        rows.append({"sürüm": name, "çalıştırma": i, **score(r)})
print(f"\nv2 bozuk JSON: {fails}")
print(pd.DataFrame(rows).to_string(index=False))


v2 bozuk JSON: 0
sürüm  çalıştırma         olay0         olay1       olay2  bulunamayan_alıntı  çözüldü/süre_iddiası  8030_ve_9000  NoRoute_kanıtı
   v1           0 medium/medium medium/medium   high/high                   1                     0         False            True
   v1           1 medium/medium medium/medium   high/high                   1                     0         False            True
   v1           2 medium/medium      low/high high/medium                   0                     0         False            True
   v1           3 medium/medium medium/medium high/medium                   0                     0         False           False
   v2           0       low/low       low/low high/medium                   0                     0         False            True
   v2           1    medium/low       low/low high/medium                   0                     0         False            True
   v2           2      none/low   none/medium high/medium               

In [15]:
# 1) LAST_RAW = Hücre 10'daki son (3.) v2 çalıştırmasının ham çıktısı
head, sep, tail = LAST_RAW.partition("</think>")
print("</think> var mı:", bool(sep), "| düşünce:", len(head), "karakter | JSON kısmı:", len(tail), "karakter")

# 2) v2 çalıştırma 0'ın üç olayı (r2)
for ev in r2["events"]:
    print(f"\n=== olay {ev['event_id']} | {ev['severity']}/{ev['confidence']} ===")
    print("özet:", ev["summary"])
    print("güven gerekçesi:", ev["confidence_reason"])
    for e in ev["evidence"]:
        print("  kanıt:", e["time"], "|", e["quote"][:110])
    print("varsayımlar:", ev["assumptions"])
    print("önerilen çözüm:", ev["recommended_fix"])
print("\nolaylar arası:", r2["cross_event_notes"])

</think> var mı: True | düşünce: 1864 karakter | JSON kısmı: 4791 karakter

=== olay 0 | low/low ===
özet: Bilinmeyen konteyner ID'si için 'Container complete event' hatası oluştu. Konteyner, önce ataması başarısız olmuş ve daha sonra tamamlandı olarak işaretlendi.
güven gerekçesi: Hata mesajı yalnızca konteynerin bilinmeyen bir ID ile tamamlandığını belirtir. Nedenin konteynerin ataması sırasında bir sorun olabileceği varsayılmaktadır.
  kanıt: 18:04:11.034 | Container complete event for unknown container id container_1445144423722_0020_01_000012
  kanıt: 18:04:11.034 | Received completed container container_1445144423722_0020_01_000012
varsayımlar: ['Konteynerin ataması sırasında bir sorun yaşanmış olabilir.', 'Sistemde konteyner takip mekanizması eksik olabilir.']
önerilen çözüm: None

=== olay 1 | low/low ===
özet: Bir konteynerin ApplicationMaster tarafından öldürülmesi raporlandı. Konteyner, önce temizlenmeye başlanmış ve ardından öldürülerek işlem tamamlanmıştır.
güven gerekçesi

In [19]:
# ==========================================
# YENİ HÜCRE: Kök Neden Analiz Modülü & Pipeline Entegrasyonu
# ==========================================

import json
import re

class QwenLocalAnalyzer:
    def __init__(self, model, tok, system_prompt):
        self.model = model
        self.tok = tok
        self.system_prompt = system_prompt

    def analyze_package(self, payload_str: str, thinking: bool = True, max_new_tokens: int = 12000):
        """
        Log paketini (JSON payload) alıp Qwen3-14B ile analiz eder.
        Düşünce zincirini (<think>) temizleyerek sadece doğrulanmış JSON döndürür.
        """
        user_content = "Analyze every event in the following package.\n\n" + payload_str
        msgs = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_content}
        ]

        text = self.tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
        inputs = self.tok(text, return_tensors="pt").to(self.model.device)

        params = dict(temperature=0.6, top_p=0.95, top_k=20) if thinking else dict(temperature=0.7, top_p=0.8, top_k=20)

        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, **params)

        gen = out[0][inputs["input_ids"].shape[1]:]
        raw_output = self.tok.decode(gen, skip_special_tokens=True).strip()

        # </think> etiketini ayıkla
        body = raw_output.split("</think>")[-1] if "</think>" in raw_output else raw_output

        # JSON objesini cımbızla (regex)
        m = re.search(r'\{\s*"events"', body)
        body = body[m.start(): body.rfind("}") + 1] if m else body[body.find("{"): body.rfind("}") + 1]

        parsed_json = json.loads(body)
        return parsed_json

# --- 1. Analyzer Örneği Oluşturma ---
analyzer_engine = QwenLocalAnalyzer(model=model, tok=tok, system_prompt=SYSTEM)

# --- 2. Analiz Sonucunu Kaydetme ve Formatlı Gösterme ---
# Colab belleğindeki hazır r2 sonucunu nihai rapor olarak kaydediyoruz
final_report = r2

with open("final_root_cause_report.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, ensure_ascii=False, indent=2)

print(" Kök Neden Analiz Raporu 'final_root_cause_report.json' olarak kaydedildi.\n")

# --- 3. Özet Konsol Çıktısı ---
print("="*80)
print(" KÖK NEDEN ANALİZİ ÖZET RAPORU (Qwen3-14B)")
print("="*80)

for ev in final_report["events"]:
    sev_color = " HIGH" if ev['severity'] == "high" else (" MEDIUM" if ev['severity'] == "medium" else " LOW")
    conf_color = " HIGH" if ev['confidence'] == "high" else (" MEDIUM" if ev['confidence'] == "medium" else " LOW")

    print(f"\n OLAY {ev['event_id']} | Önem: {sev_color} | Güven: {conf_color}")
    print(f" Özet: {ev['summary']}")
    print(f" Olası Kök Neden: {ev['likely_root_cause']}")
    print(f" Çözüm/İnceleme Önerisi: {ev.get('recommended_fix') or ev.get('recommended_checks', ['Yok'])[0]}")
    print(f" Doğrulanmış Kanıt Sayısı: {len(ev.get('evidence', []))}")
    print("-" * 50)

print(f"\n Olaylar Arası Notlar:\n{final_report.get('cross_event_notes', 'Yok')}")


 Kök Neden Analiz Raporu 'final_root_cause_report.json' olarak kaydedildi.

 KÖK NEDEN ANALİZİ ÖZET RAPORU (Qwen3-14B)

 OLAY 0 | Önem:  LOW | Güven:  LOW
 Özet: Bilinmeyen konteyner ID'si için 'Container complete event' hatası oluştu. Konteyner, önce ataması başarısız olmuş ve daha sonra tamamlandı olarak işaretlendi.
 Olası Kök Neden: Konteynerin ataması sırasında bir sorun yaşanmış ve tamamlanması sırasında sistemde takip edilemeyen bir durum oluşmuş olabilir.
 Çözüm/İnceleme Önerisi: Konteyner atama süreçlerini izleyen logları inceleyin.
 Doğrulanmış Kanıt Sayısı: 2
--------------------------------------------------

 OLAY 1 | Önem:  LOW | Güven:  LOW
 Özet: Bir konteynerin ApplicationMaster tarafından öldürülmesi raporlandı. Konteyner, önce temizlenmeye başlanmış ve ardından öldürülerek işlem tamamlanmıştır.
 Olası Kök Neden: ApplicationMaster, konteynerin işlemi tamamlanmış veya hatalı olduğunu tespit ederek onu öldürmüş olabilir.
 Çözüm/İnceleme Önerisi: ApplicationMaster'in kon